[Back to Computer Organization and Architecture guideline](Computer-Organization.html)

## **Pipelining and Hazards** {#pipelining-and-hazards}

Chapter 06 followed one instruction at a time through instruction fetch, decode, execution, memory access, and write-back. A single-cycle processor performs all of those activities during one long clock period, while a multi-cycle processor reuses hardware across several shorter periods. **Pipelining** makes a third choice: keep the phases in separate hardware regions and let different instructions occupy different phases at the same time.

This overlap changes the microarchitecture, not the ISA contract. `add x5, x6, x7` must still produce exactly the same architectural result, whether the processor is single-cycle, five-stage pipelined, or superscalar. The pipeline may contain partially executed and even speculative instructions internally, but committed register values, memory values, traps, and control flow must remain indistinguishable from sequential program execution.

### **Why Pipelining Improves Throughput** {#why-pipelining-improves-throughput}

Pipelining is analogous to an assembly line. If one worker completes all five manufacturing steps before accepting another item, the completion interval is the sum of all step times. If five specialized stations work concurrently, the first item still visits every station, but a new item can enter whenever the slowest station is ready. The gain is therefore mainly in **throughput**, the rate of completed instructions, rather than in the latency of one instruction.

![Without overlap, four instructions consume twenty stage-times; after a five-stage pipeline fills, the same four instructions consume eight stage-times. One instruction still traverses all five stages.](assets/pipeline-throughput-latency.svg){fig-align="center" width="100%"}

Let the combinational delays of the five stages be $t_{IF}$, $t_{ID}$, $t_{EX}$, $t_{MEM}$, and $t_{WB}$. A single-cycle implementation must allow approximately

$$
T_{single}\ge t_{IF}+t_{ID}+t_{EX}+t_{MEM}+t_{WB}
$$

for one instruction. After inserting pipeline registers, the clock period is instead constrained by the slowest stage plus register overhead:

$$
T_{pipe}\ge \max(t_{IF},t_{ID},t_{EX},t_{MEM},t_{WB})+t_{reg}.
$$

- $T_{single}$ is the minimum clock period of the simplified single-cycle design.
- $T_{pipe}$ is the minimum clock period of the pipelined design.
- $t_{IF}$ through $t_{WB}$ are the useful logic delays inside individual stages.
- $t_{reg}$ combines clock-to-Q delay, setup time, and clock uncertainty associated with the pipeline registers.
- The maximum appears because every stage receives the same clock period; the slowest stage sets the pace.

Pipelining is useful when a long operation can be split into reasonably balanced stages and many independent items are available. It is less effective when one stage remains much slower than the rest, register overhead is large, or dependencies repeatedly prevent new work from advancing. These limits are the reason this chapter studies hazards rather than treating a five-stage pipeline as an automatic fivefold speedup.

### **The Five-Stage Instruction Pipeline** {#the-five-stage-instruction-pipeline}

A classic teaching pipeline divides a scalar RISC instruction into **IF**, **ID**, **EX**, **MEM**, and **WB**. The division is not an architectural requirement. Real processors may split fetch, decode, execution, and memory access into many more stages, combine simple phases, or let different instruction classes use different execution pipelines. The five-stage model is valuable because it makes the timing of values and hazards visible without hiding the core ideas behind implementation detail.

::: {.diagram-scroll .wide-diagram}
![The five-stage pipeline contains the same main functions as the Chapter 06 datapath, but four pipeline registers preserve each instruction's data, destination, control, and validity between stages.](assets/five-stage-pipeline-overview.svg){fig-align="center"}
:::

| Stage | Main question | Representative work | Value sent forward |
|---|---|---|---|
| IF | Which instruction should enter? | read instruction, calculate sequential or predicted next PC | instruction bits, PC, prediction metadata |
| ID | What does the instruction request? | decode, read registers, generate immediate, detect hazards | operands, source/destination names, control |
| EX | What result, address, or decision is required? | ALU operation, address calculation, branch comparison and target | ALU result, branch outcome, store data |
| MEM | Does the instruction access data memory? | load, store, or pass an ALU result through | loaded data or previous result |
| WB | Which architectural register changes? | select result and apply `RegWrite` | committed register value |

The following idealized schedule carries the same representative instruction classes used in Chapter 06. The instructions are intentionally independent, so one enters each cycle and no correction is yet needed.

::: {.diagram-scroll .wide-diagram}
![An independent add, load, store, branch, and jal overlap across cycles. The diagonal pattern shows that one instruction advances by one stage per cycle while each stage serves a different instruction.](assets/five-instruction-pipeline-timeline.svg){fig-align="center"}
:::

The first result appears only after the pipeline fills. Thereafter, an ideal scalar pipeline completes one instruction per cycle even though each instruction spends about five cycles inside the machine. A real trace differs whenever an instruction needs a result that is not ready, two stages request one physical resource, the predicted PC is wrong, or an exception invalidates younger work.

#### **Instruction Fetch** {#instruction-fetch}

The **instruction fetch stage (IF)** uses the current program counter to request instruction bytes and calculates a candidate next PC. In the simplest pipeline, the candidate is $PC+4$ for a 32-bit RV32I instruction. With control-flow prediction, IF may instead select a predicted branch or jump target so that the front end does not wait for an older branch to reach EX.

IF must preserve more than instruction bits. The fetched instruction's own PC is needed later for PC-relative targets, `jal`'s link value, exception reporting in `mepc`, and debugging. A practical IF/ID payload therefore contains at least `(valid, PC, instruction)` and may also carry the predicted direction, predicted target, and an instruction-access fault.

An instruction-cache miss or permission fault complicates the simple one-cycle picture. The fetch interface may hold the PC, mark the response invalid, or inject an exception record until the memory hierarchy responds. Chapter 08 will explain why an instruction cache usually supplies the fast common case while preserving a slower miss path.

The PC update and IF/ID update must be coordinated. During an ordinary cycle both advance. During a data-hazard stall both are held so the same instruction remains in decode. During a branch correction the PC is redirected and the sequentially fetched IF/ID entry is invalidated. Updating only one of these state elements would either duplicate or lose an instruction.

#### **Instruction Decode** {#instruction-decode}

The **instruction decode stage (ID)** interprets opcode and function fields, reconstructs the immediate, reads `R[rs1]` and `R[rs2]`, and creates control signals for later stages. It also exposes source and destination register names to the hazard unit. Decode is therefore both a semantic stage and a scheduling checkpoint: the processor learns what the instruction means and whether it may advance safely.

For `sw x10, 4(x11)`, ID identifies `x11` as the base source, `x10` as the store-data source, reconstructs the S-type immediate 4, and sets controls that request address addition in EX and a memory write in MEM. For `beq x12, x13, L`, it identifies two comparison operands, reconstructs a B-type offset, and marks the instruction as conditional control flow.

The register values read in ID are not always the newest values. An older instruction may have calculated a replacement value but not yet written the register file. The pipeline carries the source names into ID/EX so the forwarding unit can compare them with older destination names and replace stale operand values before the ALU uses them. If no bypass can deliver the value in time, the interlock holds the instruction in ID.

Some encodings do not use every apparent register field. Hazard logic must use decoded `uses_rs1` and `uses_rs2` flags rather than blindly comparing bit positions. Otherwise, immediate bits that happen to equal an older destination register could create a false stall.

#### **Execute** {#execute}

The **execute stage (EX)** performs the instruction's main combinational transformation. Depending on the decoded controls, the ALU adds two registers, combines a register and an immediate, calculates a load/store effective address, or forms a PC-relative target. A branch comparator determines whether a condition such as equality is true.

For a load,

$$
EA=(R[rs1]+Imm)\bmod 2^{32},
$$

where $EA$ is the effective byte address, $R[rs1]$ is the base value, and $Imm$ is the sign-extended offset. The modulo reflects RV32I's 32-bit address arithmetic. For a conditional branch, EX typically produces both a Boolean outcome and a target:

$$
taken=(A=B), \qquad target=PC+Imm
$$

for `beq`. Here $A$ and $B$ are the comparison operands after forwarding. The saved instruction PC, not the current fetch PC, must be used because several younger instructions may already have entered the pipeline.

EX is the central consumer of bypassed data. Forwarding multiplexers choose each ALU input from the original ID/EX operand, a newer result in EX/MEM, or a value approaching write-back in MEM/WB. Resolving branches in EX keeps decode simple but allows two younger instructions to enter IF and ID before the outcome is known. Earlier resolution reduces branch penalty at the cost of additional comparison, target, and forwarding logic in an earlier stage.

#### **Memory Access** {#memory-access}

The **memory access stage (MEM)** uses the EX address for loads and stores. A load requests bytes, applies size and sign/zero-extension rules, and sends the resulting word toward WB. A store combines the address with forwarded store data and byte-enable signals. An arithmetic instruction does not access data memory; its ALU result simply passes through the EX/MEM and MEM/WB registers.

This stage is where memory-side effects must be carefully controlled. A store may be present internally but must not update memory if its valid bit is zero, an older exception redirects execution, or its own access raises a fault. A precise in-order pipeline can delay the store enable until the instruction is known to be valid and older instructions cannot later invalidate it.

The one-cycle MEM abstraction assumes a cache hit. If the data cache misses, a load result does not exist at the end of the expected cycle. The pipeline must stall dependent work or use a more general request/response mechanism. The memory hierarchy therefore affects both the clock period and the average CPI; that connection becomes central in Chapter 08.

#### **Write-Back** {#write-back}

The **write-back stage (WB)** selects the value that becomes visible in the architectural register file. An arithmetic instruction selects its ALU result, a load selects returned memory data, and `jal` selects the saved link value $PC+4$. The write occurs only when the entry is valid, `RegWrite=1`, no suppressing exception is present, and $rd\ne0$.

$$
R^{+}[rd]=
\begin{cases}
WBValue, & valid\land RegWrite\land(rd\ne0),\\
R[rd], & \text{otherwise}.
\end{cases}
$$

- $WBValue$ is the result selected by the write-back multiplexer.
- `valid` says that this pipeline entry still represents a real instruction.
- `RegWrite` comes from decode and has travelled with the instruction.
- $rd\ne0$ preserves RISC-V's hardwired zero register.

WB is late enough to provide a simple ordered commit point for register results: older instructions reach WB before younger ones. However, consumers should not wait for this point when a result was already calculated in EX or returned from MEM. Forwarding improves timing without changing the architectural moment at which the register file is updated.

### **Pipeline Registers and Timing** {#pipeline-registers-and-timing}

Pipeline registers are not merely separators. Each one stores the complete **in-flight record** needed by the next stages: values, register names, control bits, the instruction PC, exception metadata, and a valid bit. If data and control become misaligned, one instruction can accidentally use another instruction's destination or memory-write enable.

![Each pipeline register carries the fields still needed downstream. Fields disappear after their final consumer, while valid, destination, control, and exception information travel as far as required.](assets/pipeline-register-payloads.svg){fig-align="center" width="100%"}

| Register | Typical payload | Why it is retained |
|---|---|---|
| IF/ID | valid, PC, instruction, prediction/fetch status | decode meaning and identify the instruction |
| ID/EX | operands, `rs1`, `rs2`, `rd`, immediate, PC, controls | execute with hazard-aware values |
| EX/MEM | ALU result, branch outcome/target, store data, `rd`, controls | access memory or redirect control flow |
| MEM/WB | loaded data or ALU/link result, `rd`, `RegWrite` | perform final register update |

The clock must cover register output delay, the slowest combinational path between adjacent registers, setup time, and clock uncertainty:

$$
T_{clk}\ge t_{cq}+\max_i(t_{logic,i})+t_{setup}+t_{skew}.
$$

- $T_{clk}$ is the common pipeline clock period.
- $t_{cq}$ is the time from the active clock edge until a source register's output is valid.
- $t_{logic,i}$ is the combinational delay through stage path $i$.
- $t_{setup}$ is how long the destination input must be stable before the next edge.
- $t_{skew}$ represents clock arrival mismatch and design margin.

Stage balancing matters because the maximum, not the average, controls the clock. Splitting a 400 ps stage into two 200 ps stages can improve frequency, but it adds another pipeline register, increases instruction latency, creates more in-flight state, and can increase the number of wrong-path stages flushed after a branch.

Three control actions are easy to confuse:

| Action | State update | Meaning |
|---|---|---|
| normal advance | every stage captures its predecessor | all valid instructions move one stage |
| stall | selected registers hold old values | an instruction waits; older stages may continue |
| bubble | destination register captures `valid=0` or zero side-effect controls | an empty slot moves through the pipeline |
| flush | one or more younger entries are invalidated | wrong-path or post-exception work is discarded |

<details>
<summary>Python model: valid bits make advance, stall, bubble, and flush explicit</summary>

```python
from dataclasses import dataclass


@dataclass(frozen=True)
class PipeEntry:
    valid: bool = False
    text: str = "bubble"


def update_front_end(pc, if_id, fetched, *, stall=False, flush=False, target=None):
    """Model only the PC and IF/ID state changed at one clock edge."""
    if flush:
        # Redirect fetch and invalidate the already-fetched wrong-path entry.
        return target, PipeEntry()
    if stall:
        # Holding both values prevents duplication or loss.
        return pc, if_id
    return pc + 4, fetched


pc = 0x100
if_id = PipeEntry(True, "add x5, x6, x7")

held_pc, held_if_id = update_front_end(pc, if_id, PipeEntry(), stall=True)
assert (held_pc, held_if_id) == (pc, if_id)

new_pc, cleared_if_id = update_front_end(
    pc, if_id, PipeEntry(), flush=True, target=0x180
)
assert new_pc == 0x180 and not cleared_if_id.valid
```

</details>

### **Pipeline Performance** {#pipeline-performance}

Performance analysis separates the ideal overlap from the events that leave slots empty. For $N$ independent instructions in a $k$-stage pipeline with a common period $T_{clk}$, the ideal execution time is

$$
T_{ideal}(N)=(k+N-1)T_{clk}.
$$

- $N$ is the number of instructions in the measured sequence.
- $k$ is the number of pipeline stages.
- The first $k$ cycles fill the pipeline and complete the first instruction.
- Each of the remaining $N-1$ instructions ideally completes one cycle later.
- $T_{clk}$ converts the cycle count into time.

For a five-stage pipeline and five independent instructions, $k+N-1=5+5-1=9$ cycles, matching the idealized diagonal schedule above. For a long sequence, the four fill cycles become a small fraction of total time. Stalls and flushes add extra cycles and are not hidden by the formula.

#### **Latency and Throughput** {#latency-and-throughput}

**Instruction latency** is the time from an instruction entering IF until it completes. Under the ideal fixed-stage model,

$$
L_{instruction}\approx kT_{clk}.
$$

**Steady-state throughput** is the completion rate after the pipeline fills:

$$
Throughput_{ideal}\approx \frac{1}{T_{clk}}\quad\text{instruction per unit time}.
$$

These formulas describe different questions. Increasing $k$ can shorten each stage and increase throughput, yet the extra register overhead may keep $kT_{clk}$ unchanged or even increase it. A five-stage processor that has a 250 ps clock gives one ideal completion every 250 ps, but one instruction spends about $5\times250=1250$ ps traversing the stages.

| Metric | Question | Pipelining effect |
|---|---|---|
| latency | How long does one instruction take from entry to completion? | may stay similar or increase |
| throughput | How often can instructions complete? | usually improves after fill |
| clock period | How long must one cycle be? | determined by slowest stage plus register overhead |
| response to a dependency | Can the next instruction advance now? | may insert bubbles and reduce effective throughput |

This distinction prevents a common misconception: seeing five active stages does not mean each instruction finishes five times sooner. It means the machine is using those five stage resources concurrently on different instructions.

#### **Ideal Speedup** {#ideal-speedup}

Suppose the corresponding unpipelined execution performs all stage work sequentially for every instruction. Its simplified time is

$$
T_{unpipe}(N)=N\sum_{i=1}^{k}t_i.
$$

Combining this with the ideal pipeline time gives

$$
Speedup(N)=
\frac{N\sum_{i=1}^{k}t_i}
{(k+N-1)(\max_i t_i+t_{reg})}.
$$

- $t_i$ is the useful delay of stage $i$ before register overhead.
- $\sum t_i$ is the total useful work on the unpipelined path.
- $\max_i t_i+t_{reg}$ is the pipeline period in this simplified model.
- $k+N-1$ includes fill and drain effects.

If all stages are perfectly balanced, register overhead is negligible, and $N$ is very large, the speedup approaches $k$. Real speedup is lower because the stages are unequal, pipeline registers consume time and energy, and hazards add cycles. The ideal bound is a design target, not a guaranteed outcome.

<details>
<summary>Python example: quantify stage imbalance and register overhead</summary>

```python
def ideal_pipeline_speedup(stage_delays_ps, register_overhead_ps, instructions):
    """Compare sequential stage work with an ideal filled pipeline."""
    k = len(stage_delays_ps)
    unpipelined_time = instructions * sum(stage_delays_ps)
    pipeline_clock = max(stage_delays_ps) + register_overhead_ps
    pipelined_time = (k + instructions - 1) * pipeline_clock
    return {
        "clock_ps": pipeline_clock,
        "unpipelined_ps": unpipelined_time,
        "pipelined_ps": pipelined_time,
        "speedup": unpipelined_time / pipelined_time,
    }


result = ideal_pipeline_speedup(
    stage_delays_ps=[180, 150, 220, 200, 100],
    register_overhead_ps=20,
    instructions=100,
)

assert result["clock_ps"] == 240
assert round(result["speedup"], 2) == 3.41
print(result)
```

</details>

The example contains five stages but achieves only about $3.41\times$ ideal sequence speedup for 100 instructions. The 220 ps EX stage determines the 240 ps clock, so shorter stages have unused timing slack. Register overhead and four fill cycles reduce the gain further.

#### **CPI with Stalls** {#cpi-with-stalls}

**Cycles per instruction (CPI)** measures how many clock cycles are consumed per completed instruction on average. If $S$ extra stall or flush cycles occur while $N$ instructions complete in a $k$-stage scalar pipeline,

$$
Cycles=(k-1)+N+S,
$$

$$
CPI=\frac{(k-1)+N+S}{N}.
$$

- $k-1$ is the fill/drain overhead beyond the one ideal cycle assigned to each instruction.
- $N$ counts completed architectural instructions; flushed wrong-path instructions do not count.
- $S$ is every additional cycle caused by unavailable values, resource conflicts, branch corrections, or cache delays.
- For a long sequence, $(k-1)/N$ approaches zero, so $CPI\approx1+S/N$.

Execution time remains the product introduced in Chapter 01:

$$
CPU\ Time=N\times CPI\times T_{clk}.
$$

A deeper pipeline can reduce $T_{clk}$ while increasing misprediction penalty and therefore CPI. Comparing clock frequency alone can select the slower processor.

<details>
<summary>Python example: combine load-use, branch, and cache penalties</summary>

```python
def pipeline_cpi(
    instructions,
    stages,
    load_use_events=0,
    load_use_penalty=1,
    branch_misses=0,
    branch_penalty=2,
    cache_misses=0,
    cache_penalty=0,
):
    """Account for fill cycles and independent sources of lost cycles."""
    stalls = (
        load_use_events * load_use_penalty
        + branch_misses * branch_penalty
        + cache_misses * cache_penalty
    )
    cycles = (stages - 1) + instructions + stalls
    return cycles, cycles / instructions


cycles, cpi = pipeline_cpi(
    instructions=1000,
    stages=5,
    load_use_events=40,
    branch_misses=25,
    cache_misses=5,
    cache_penalty=12,
)

assert cycles == 1154
assert round(cpi, 3) == 1.154
print(f"cycles={cycles}, CPI={cpi:.3f}")
```

</details>

The code adds penalties for clarity, but real events may overlap. A cache miss can already hold a branch or a dependent instruction, so a cycle-accurate simulator must model state rather than blindly sum independent rates. The additive model is most useful as a first-order explanation of where CPI above one comes from.

### **Structural Hazards** {#structural-hazards}

A **structural hazard** occurs when simultaneous pipeline stages request more instances or ports of a hardware resource than the implementation provides. It is a capacity problem. The instructions need not share a register value or control-flow relationship.

The classic example is a unified, single-port memory. IF wants to read the next instruction every cycle, while a load or store in MEM wants to access data during the same cycle. A single port can accept only one request, so the implementation must provide more capacity or delay one requester.

![Instruction fetch and a load in MEM conflict for one unified memory port. Separate instruction and data caches, additional ports or banks, or a stall make the implementation correct.](assets/structural-hazard-memory-conflict.svg){fig-align="center" width="100%"}

Common structural choices include:

| Conflict | Hardware solution | Scheduling solution | Trade-off |
|---|---|---|---|
| IF and MEM need memory | separate instruction/data caches or two ports | stall fetch or MEM | capacity costs area; stalling costs CPI |
| WB writes while ID reads registers | multiported register file and defined same-cycle timing | stall or add write-to-read bypass | ports increase array delay and wiring |
| EX needs both branch target and arithmetic | dedicated target adder/comparator | reuse ALU across cycles | duplication improves throughput |
| two-wide issue needs two ALUs | duplicate execution units | issue only one instruction | width is useful only when resources exist |

<details>
<summary>Python model: detect competing reservations for one-cycle resources</summary>

```python
from collections import Counter


def structural_conflicts(stage_requests, capacities):
    """Return resources requested more times than their cycle capacity."""
    demand = Counter(stage_requests.values())
    return {
        resource: {"demand": count, "capacity": capacities.get(resource, 0)}
        for resource, count in demand.items()
        if count > capacities.get(resource, 0)
    }


requests = {
    "IF": "unified_memory_port",
    "MEM": "unified_memory_port",  # a load is in MEM this cycle
    "EX": "integer_alu",
}

conflicts = structural_conflicts(
    requests,
    capacities={"unified_memory_port": 1, "integer_alu": 1},
)

assert conflicts["unified_memory_port"] == {"demand": 2, "capacity": 1}
print(conflicts)
```

</details>

Eliminating all structural stalls is not always optimal. A rare conflict may be cheaper to schedule around than to duplicate a large resource. The correct engineering question is whether the performance saved by extra capacity justifies its area, energy, timing, and verification cost.

### **Data Hazards** {#data-hazards}

A **data dependence** is a property of the program: one instruction's meaning is related to another instruction's read or write. A **data hazard** is a microarchitectural timing problem: overlap would let the operations occur in an order that violates that dependence. The distinction matters because the same program dependence may require a stall in one pipeline, forwarding in another, and no special action in a single-cycle implementation.

Consider:

```text
I0: add x5, x1, x2
I1: sub x7, x5, x3
```

`I1` truly needs the value produced by `I0`. Without overlap, `I0` writes `x5` before `I1` reads it. In a five-stage pipeline, `I1` can reach EX while `I0` has not yet reached WB. The hazard question is therefore:

> At which stage is the producer's value available, and at which stage does the consumer need it?

| Value | Normally available | Typical consumer need | Consequence |
|---|---|---|---|
| integer ALU result | end of producer EX | start of consumer EX | forward EX/MEM to EX |
| load result on a hit | end of producer MEM | start of consumer EX | immediate consumer must stall once |
| store data | carried toward MEM | producer result may arrive before store MEM | add store-data forwarding |
| branch operand | branch comparison stage | often EX in this design | use ALU-style forwarding or stall |

Hazard handling should use availability and need times, not memorize isolated instruction pairs. That timing view scales to multipliers, cache misses, floating-point units, and out-of-order execution.

#### **RAW, WAR, and WAW Dependencies** {#raw-war-and-waw-dependencies}

Register dependencies are classified by the order of reads (**R**) and writes (**W**) to the same architectural name.

![RAW carries a real value from producer to consumer. WAR and WAW preserve ordering between uses of the same register name and become hazards when execution or completion may be reordered.](assets/dependency-types.svg){fig-align="center" width="100%"}

| Type | Required program order | Meaning | Five-stage in-order scalar pipeline |
|---|---|---|---|
| RAW | earlier write before later read | true/flow dependence; a value is communicated | can be a hazard; forward or stall |
| WAR | earlier read before later write | anti-dependence caused by reusing a name | not a hazard when reads and writes remain ordered |
| WAW | earlier write before later write | output dependence caused by reusing a name | not a hazard with one ordered WB path |

RAW cannot be removed without changing the algorithm because the later instruction needs the earlier result. WAR and WAW are **name dependencies**: two instructions happen to use the same architectural register even though no value flows from the earlier one to the later one. Register renaming gives the writes different physical destinations and removes these false constraints in out-of-order processors.

<details>
<summary>Python model: classify dependencies from read and write sets</summary>

```python
from dataclasses import dataclass


@dataclass(frozen=True)
class RegisterUse:
    text: str
    reads: frozenset[str]
    writes: frozenset[str]


def dependencies(older, younger):
    """Classify dependencies from the older instruction to the younger one."""
    return {
        "RAW": older.writes & younger.reads,
        "WAR": older.reads & younger.writes,
        "WAW": older.writes & younger.writes,
    }


add = RegisterUse("add x5,x1,x2", frozenset({"x1", "x2"}), frozenset({"x5"}))
sub = RegisterUse("sub x7,x5,x3", frozenset({"x5", "x3"}), frozenset({"x7"}))

result = dependencies(add, sub)
assert result["RAW"] == frozenset({"x5"})
assert not result["WAR"] and not result["WAW"]
print(result)
```

</details>

Memory dependencies need address information rather than register names. A store followed by a load from the same address has a RAW-like memory dependence, but the processor may not know whether their addresses match until EX. Memory disambiguation becomes a major challenge in speculative out-of-order designs.

#### **Forwarding** {#forwarding}

**Forwarding**, also called **bypassing**, sends a completed result directly from a later pipeline register to the stage that needs it, instead of waiting for register-file write-back. It changes the physical route and timing of a value but does not change program order or architectural register semantics.

::: {.diagram-scroll .wide-diagram}
![The forwarding unit compares the consumer's rs1 and rs2 with destination registers in EX/MEM and MEM/WB. The nearest matching producer receives priority because it contains the newest value.](assets/pipeline-forwarding-network.svg){fig-align="center"}
:::

For ALU input A, a simplified priority rule is:

$$
ForwardA=
\begin{cases}
EXMEM, & EXMEM.RegWrite\land(EXMEM.rd\ne0)\land(EXMEM.rd=IDEX.rs1),\\
MEMWB, & MEMWB.RegWrite\land(MEMWB.rd\ne0)\land(MEMWB.rd=IDEX.rs1),\\
IDEX, & \text{otherwise}.
\end{cases}
$$

- `IDEX.rs1` names the source needed by the current EX-stage consumer.
- `EXMEM.rd` belongs to the nearest older instruction and therefore has priority.
- `MEMWB.rd` belongs to an older producer used only when EX/MEM does not match.
- `RegWrite` prevents a branch or store's incidental `rd` bits from being treated as a result.
- The $rd\ne0$ test prevents forwarding a supposed write to RISC-V register `x0`.

The same rule is applied to `rs2` for ALU operand B. Store data may need its own bypass path because a store consumes that value in MEM rather than as the address ALU's second input.

<details>
<summary>Python model: select the newest forwarded operand</summary>

```python
from dataclasses import dataclass


@dataclass(frozen=True)
class Producer:
    reg_write: bool
    rd: int
    value: int


def forwarded_operand(source_reg, register_value, ex_mem, mem_wb):
    """Use the nearest valid producer; otherwise keep the register value."""
    if ex_mem.reg_write and ex_mem.rd != 0 and ex_mem.rd == source_reg:
        return ex_mem.value, "EX/MEM"
    if mem_wb.reg_write and mem_wb.rd != 0 and mem_wb.rd == source_reg:
        return mem_wb.value, "MEM/WB"
    return register_value, "ID/EX"


# add x5,... is in EX/MEM while sub ...,x5,... enters EX.
value, source = forwarded_operand(
    source_reg=5,
    register_value=19,              # stale x5 read in ID
    ex_mem=Producer(True, 5, 42),   # newest add result
    mem_wb=Producer(True, 5, 31),   # an even older x5 result
)

assert (value, source) == (42, "EX/MEM")
print(value, source)
```

</details>

Forwarding solves a timing gap only when the value already exists. An ALU result is available at the end of EX and can feed the next instruction's EX inputs in the following cycle. A load value appears later, at the end of MEM, so the immediately following consumer reaches EX too soon. That case requires an interlock.

#### **Stalling and Interlocks** {#stalling-and-interlocks}

A **stall** prevents selected instructions from advancing. An **interlock** is hardware that detects the unsafe condition and generates the hold and bubble controls automatically. Correct software therefore does not need to insert implementation-specific no-operations around ordinary dependencies.

For the load-use pair

```text
lw  x5, 0(x6)
add x7, x5, x8
```

the load address is calculated in EX during cycle C3, but the data is returned only at the end of MEM in C4. The add would normally begin EX in C4 and needs `x5` at that stage's input. Even a MEM-to-EX bypass cannot deliver a value before it has been produced. The pipeline delays the add by one cycle, after which the returned load value can be forwarded.

::: {.diagram-scroll .wide-diagram}
![The load advances to MEM while PC and IF/ID are held. Clearing the ID/EX controls inserts one bubble, so the dependent add reaches EX one cycle later and receives the load result by forwarding.](assets/load-use-stall-bubble.svg){fig-align="center"}
:::

A common detection equation is

$$
stall=IDEX.MemRead\land(IDEX.rd\ne0)\land
\left[
(uses\_rs1\land IDEX.rd=IFID.rs1)\lor
(uses\_rs2\land IDEX.rd=IFID.rs2)
\right].
$$

- `IDEX.MemRead` says the older EX-stage instruction is a load whose result is late.
- `IDEX.rd` is that load's destination.
- `IFID.rs1` and `IFID.rs2` are source names of the consumer waiting in decode.
- `uses_rs1` and `uses_rs2` suppress comparisons for operands the decoded instruction does not actually read.

When the expression is true, the hazard unit holds the PC, holds IF/ID, clears the side-effect controls entering ID/EX, and lets EX/MEM and later stages advance. The load moves forward, a bubble occupies EX, and the consumer remains available for another decode attempt.

<details>
<summary>Python model: load-use detection and the exact interlock response</summary>

```python
def load_use_hazard(load, consumer):
    """Return True only when a decoded source needs the EX-stage load result."""
    if not load["mem_read"] or load["rd"] == 0:
        return False
    needs_rs1 = consumer["uses_rs1"] and consumer["rs1"] == load["rd"]
    needs_rs2 = consumer["uses_rs2"] and consumer["rs2"] == load["rd"]
    return needs_rs1 or needs_rs2


load = {"mem_read": True, "rd": 5}
consumer = {"uses_rs1": True, "rs1": 5, "uses_rs2": True, "rs2": 8}

stall = load_use_hazard(load, consumer)
control = {
    "PCWrite": not stall,       # False: hold the fetch address
    "IFIDWrite": not stall,     # False: keep the consumer in decode
    "IDEXControls": "zero" if stall else "decoded",  # inject a bubble
}

assert stall
assert control == {"PCWrite": False, "IFIDWrite": False, "IDEXControls": "zero"}
print(control)
```

</details>

| Technique | Use when | Cost |
|---|---|---|
| forwarding | producer value exists before consumer needs it | multiplexers, comparators, longer bypass wiring |
| one-cycle interlock | value will exist after a short fixed delay | one lost issue slot |
| variable stall | memory or execution latency is not fixed | more general ready/valid control and potentially many lost cycles |
| compiler scheduling | an independent instruction can fill the gap | depends on available independent work and target pipeline knowledge |

### **Control Hazards** {#control-hazards}

A **control hazard** occurs because fetch needs the next PC before an older branch, jump, return, or trap has fully determined that PC. Waiting for every decision is correct but wastes instruction-fetch opportunities. Predicting a path keeps the pipeline busy, but the processor must be able to erase all effects of a wrong prediction.

For a branch resolved in EX, the sequential fetch unit can admit two younger instructions before the outcome is known: one reaches ID and one remains in IF. If the branch is taken while IF predicted $PC+4$, both younger instructions belong to the wrong path. The penalty depends on where the branch resolves, how quickly the target is available, and how many wrong-path stages must be cleared.

`jal` is unconditional, so decode can often identify that control will change before a conditional branch's register comparison is complete. Returns are harder: their targets come from registers and benefit from a return-address stack. Indirect calls and jumps require target prediction in addition to direction prediction.

#### **Pipeline Flushes** {#pipeline-flushes}

A **pipeline flush** invalidates in-flight instructions that must no longer affect architectural state. On a taken branch that was predicted not taken, the processor selects the target PC and clears valid bits for younger sequential instructions. Older instructions and the branch itself continue normally.

![When the branch in EX resolves taken, the sequential instructions in ID and IF are younger and wrong-path. The control unit invalidates them and redirects fetch to the target.](assets/control-hazard-flush.svg){fig-align="center" width="100%"}

Flushing is stronger than merely ignoring a computed result. Every possible side effect of an invalid entry must be gated:

$$
RegWrite_{effective}=valid\land RegWrite,
$$

$$
MemWrite_{effective}=valid\land MemWrite.
$$

The same validity rule suppresses wrong-path exceptions. For example, a speculative load on an incorrectly predicted path may encounter an internal access problem, but it cannot raise an architecturally visible trap after the older branch proves that the load should never have executed.

<details>
<summary>Python model: redirect the PC and squash only younger entries</summary>

```python
from dataclasses import dataclass, replace


@dataclass(frozen=True)
class InFlight:
    age: int
    text: str
    valid: bool = True


def resolve_taken_branch(branch_age, target, entries):
    """Keep the branch and older work; invalidate younger wrong-path work."""
    corrected = [
        replace(entry, valid=False) if entry.age > branch_age else entry
        for entry in entries
    ]
    return target, corrected


entries = [
    InFlight(0, "older add"),
    InFlight(1, "beq x5,x0,L"),
    InFlight(2, "wrong-path sub"),
    InFlight(3, "wrong-path add"),
]

pc, entries = resolve_taken_branch(1, 0x200, entries)
assert pc == 0x200
assert [entry.valid for entry in entries] == [True, True, False, False]
```

</details>

RV32I does not expose architectural branch delay slots: software semantics do not require the instruction after a taken branch to execute. Any wrong-path work is an internal speculation that the implementation must remove transparently.

#### **Static Prediction** {#static-prediction}

**Static prediction** chooses a direction using a fixed rule rather than per-branch runtime history. The simplest pipeline predicts every conditional branch as not taken and continues fetching sequentially. This is inexpensive because $PC+4$ is already available, and only taken branches require correction.

| Static policy | Rationale | Typical weakness |
|---|---|---|
| stall until resolved | never fetch a wrong-path instruction | loses cycles for every branch |
| always not taken | straight-line code is common and sequential PC is easy | repeated taken loop branches mispredict |
| always taken | useful when target can be supplied early | forward conditionals are often not taken |
| backward taken, forward not taken | backward branches often close loops | code structure is only a heuristic |
| compiler hint/profile decision | uses offline knowledge | behavior may change with inputs |

For a long instruction stream, a first-order branch contribution to CPI is

$$
CPI=CPI_{base}+f_b\times m\times P.
$$

- $CPI_{base}$ is CPI without branch mispredictions.
- $f_b$ is the fraction of completed instructions that are conditional branches.
- $m$ is the fraction of those branches predicted incorrectly.
- $P$ is the number of lost cycles per misprediction.

If 20% of instructions are branches, 35% are mispredicted, and two cycles are lost per miss, the added CPI is $0.20\times0.35\times2=0.14$.

<details>
<summary>Python example: compare static branch policies on one outcome trace</summary>

```python
def static_accuracy(outcomes, policy):
    """Measure always-taken or always-not-taken on T/N outcomes."""
    prediction = "T" if policy == "taken" else "N"
    correct = sum(outcome == prediction for outcome in outcomes)
    return correct / len(outcomes)


# Four taken loop-back branches followed by the not-taken loop exit.
loop = list("TTTTN")

assert static_accuracy(loop, "taken") == 0.8
assert static_accuracy(loop, "not_taken") == 0.2

branch_fraction = 0.20
misprediction_rate = 0.35
penalty = 2
added_cpi = branch_fraction * misprediction_rate * penalty
assert round(added_cpi, 2) == 0.14
print(f"added CPI = {added_cpi:.2f}")
```

</details>

Static methods are attractive in very small cores because their cost and timing are predictable. They cannot adapt when two executions of the same branch behave differently, which motivates dynamic history.

#### **Dynamic Branch Prediction** {#dynamic-branch-prediction}

**Dynamic prediction** records recent runtime behavior and uses it to predict future control flow. A basic branch history table (BHT) indexes a small state value with bits from the branch PC. A two-bit saturating counter is common because one unusual outcome weakens a strong prediction without immediately reversing it.

![A two-bit counter has strongly and weakly not-taken states and weakly and strongly taken states. Taken increments the saturated counter; not taken decrements it.](assets/two-bit-branch-predictor.svg){fig-align="center" width="100%"}

Let the counter $c\in\{0,1,2,3\}$. States 0 and 1 predict not taken; states 2 and 3 predict taken. Its update is

$$
c^{+}=\begin{cases}
\min(3,c+1), & outcome=taken,\\
\max(0,c-1), & outcome=not\ taken.
\end{cases}
$$

- $c$ is the state before the resolved branch updates the predictor.
- $c^{+}$ is the next state stored for a future encounter.
- `min` and `max` make the counter saturate instead of wrapping from strong taken to strong not taken.

Direction alone is insufficient for a taken prediction. A **branch target buffer (BTB)** caches likely target addresses so IF can redirect immediately. A **return-address stack (RAS)** predicts function-return targets, which follow nested call structure more accurately than a general BTB. Modern predictors also combine local and global histories, but they preserve the same conceptual split: predict whether control changes, predict where it goes, and verify later.

<details>
<summary>Python model: a PC-indexed two-bit predictor with aliasing</summary>

```python
class TwoBitPredictor:
    def __init__(self, entries=16, initial_state=1):
        self.counters = [initial_state] * entries  # 1 = weakly not taken

    def _index(self, pc):
        # Drop two alignment bits, then wrap into the finite table.
        return (pc >> 2) % len(self.counters)

    def predict(self, pc):
        return self.counters[self._index(pc)] >= 2

    def update(self, pc, taken):
        index = self._index(pc)
        old = self.counters[index]
        self.counters[index] = min(3, old + 1) if taken else max(0, old - 1)


predictor = TwoBitPredictor(entries=8)
pc = 0x100
outcomes = [True, True, True, True, False]
predictions = []

for taken in outcomes:
    predictions.append(predictor.predict(pc))
    predictor.update(pc, taken)

assert predictions == [False, True, True, True, True]
assert predictor.counters[predictor._index(pc)] == 2  # weakly taken after exit
print(predictions)
```

</details>

Different static branches can map to the same finite BHT entry; this **aliasing** lets one branch's outcome disturb another branch's state. Larger tables and richer histories improve accuracy but consume storage, energy, and front-end timing. A predictor is useful only if it returns a direction and target early enough to guide fetch.

| Predictor | State | Adapts to input behavior? | Main cost |
|---|---|---:|---|
| always not taken | none | no | taken-branch flushes |
| backward taken/forward not | branch displacement sign | no | heuristic errors |
| two-bit BHT | counter per indexed entry | yes | warm-up and aliasing |
| history-based predictor | counters indexed by local/global patterns | yes | more storage, latency, and update logic |

### **Exceptions in a Pipeline** {#exceptions-in-a-pipeline}

Chapter 06 defined a **precise exception** as a clean architectural boundary: every older instruction has completed, the faulting instruction has not performed its ordinary side effects, and no younger instruction has changed architectural state. Pipelining makes this harder because all three groups can be active at once.

![When a load faults in MEM, older work may complete, the load suppresses normal write-back and records trap state, younger entries are invalidated, and fetch redirects to mtvec.](assets/precise-exception-pipeline.svg){fig-align="center" width="100%"}

Each pipeline entry therefore carries its PC, valid bit, and any detected exception metadata forward until control can establish the correct age order. An illegal instruction may be detected in ID, a branch-target problem in EX, and a data access fault in MEM. If several instructions report conditions in one cycle, the architecturally oldest valid instruction must win; a younger fault may disappear when an older branch misprediction or exception flushes it.

A simple in-order response is:

1. allow instructions older than the selected fault to finish any permitted architectural updates;
2. suppress `RegWrite` or `MemWrite` for the faulting instruction;
3. write trap state such as `mepc`, `mcause`, and `mtval`;
4. invalidate every younger pipeline entry;
5. redirect the fetch PC to `mtvec`.

Interrupts are asynchronous requests rather than faults caused by the current instruction. The processor still presents a precise boundary, commonly taking the interrupt between completed instructions and recording the next resumable PC.

<details>
<summary>Python model: select the oldest valid exception and derive pipeline actions</summary>

```python
from dataclasses import dataclass


@dataclass(frozen=True)
class PipelineStatus:
    age: int
    pc: int
    text: str
    valid: bool = True
    exception: str | None = None


def precise_trap(entries, mtvec):
    """Choose the oldest valid exception and classify all in-flight work."""
    faults = [entry for entry in entries if entry.valid and entry.exception]
    if not faults:
        return None

    fault = min(faults, key=lambda entry: entry.age)
    return {
        "next_pc": mtvec,
        "mepc": fault.pc,
        "mcause": fault.exception,
        "commit_older": [entry.text for entry in entries if entry.valid and entry.age < fault.age],
        "suppress_faulting": fault.text,
        "flush_younger": [entry.text for entry in entries if entry.valid and entry.age > fault.age],
    }


snapshot = [
    PipelineStatus(0, 0x100, "older add in WB"),
    PipelineStatus(1, 0x104, "faulting load in MEM", exception="load access fault"),
    PipelineStatus(2, 0x108, "younger branch in EX"),
    PipelineStatus(3, 0x10C, "younger add in ID", exception="illegal instruction"),
]

action = precise_trap(snapshot, mtvec=0x80000000)
assert action["mepc"] == 0x104
assert action["commit_older"] == ["older add in WB"]
assert action["flush_younger"] == ["younger branch in EX", "younger add in ID"]
print(action)
```

</details>

| Event | Why work is invalidated | Is the triggering instruction committed normally? | Restart information |
|---|---|---:|---|
| branch misprediction | prediction differed from resolved control flow | yes, the branch is valid | resolved target or sequential PC |
| synchronous exception | current instruction cannot complete normally | no ordinary effects from faulting instruction | faulting PC and cause |
| interrupt | external event is accepted at an instruction boundary | all older work completes | next resumable PC and cause |

Precise state is the invariant shared by a five-stage core and a much more speculative processor. Wider and out-of-order machines need explicit retirement machinery because execution completion order no longer guarantees architectural order.

### **From Scalar Pipelines to Superscalar Execution** {#from-scalar-pipelines-to-superscalar-execution}

The five-stage design is **scalar**: at most one instruction enters each stage per cycle, so its ideal steady-state completion rate is one instruction per cycle. A **superscalar** processor has an issue width greater than one and attempts to begin multiple instructions in the same cycle. Pipeline depth and issue width are separate dimensions: depth overlaps stages across time, while width duplicates or shares resources so multiple instructions can use a stage concurrently.

![A two-wide front end considers an instruction pair, checks same-cycle dependencies and capacity, dispatches to multiple execution lanes, and preserves in-order retirement even when internal execution is more flexible.](assets/scalar-to-superscalar.svg){fig-align="center" width="100%"}

The achieved rate is often described with **instructions per cycle (IPC)**:

$$
IPC=\frac{Instructions\ retired}{Cycles},
\qquad
CPI=\frac{Cycles}{Instructions\ retired}=\frac{1}{IPC}.
$$

The reciprocal relation applies when both metrics describe the same completed instruction stream and interval. A two-wide design has a peak IPC of 2, but it reaches that peak only when useful independent instructions, execution units, register-file ports, cache bandwidth, and correct control-flow predictions are all available.

New problems appear inside one issue group. If slot 1 reads the destination of slot 0, it cannot simply execute in parallel unless the implementation supports the required same-cycle bypass or schedules it later. Two loads may compete for one data-cache port. Wider bypass networks compare many producers and consumers, and their wiring can become a clock-limiting structure.

Out-of-order superscalar processors add several mechanisms:

| Mechanism | Problem addressed |
|---|---|
| register renaming | removes WAR and WAW name dependencies |
| issue queue / scheduler | waits until true operands and an execution unit are ready |
| multiple functional units | supplies structural capacity for parallel execution |
| reorder buffer | retires results in program order and maintains precise exceptions |
| load/store queue | checks memory ordering and forwards store data to loads |

<details>
<summary>Python model: decide whether two instructions can issue together</summary>

```python
from dataclasses import dataclass


@dataclass(frozen=True)
class IssueInstruction:
    text: str
    reads: frozenset[str]
    writes: frozenset[str]
    resource: str


def can_issue_pair(first, second, capacities):
    """Check a simple in-order two-wide issue group without register renaming."""
    reasons = []

    if first.writes & second.reads:
        reasons.append("same-group RAW dependency")
    if first.reads & second.writes:
        reasons.append("same-group WAR dependency")
    if first.writes & second.writes:
        reasons.append("same-group WAW dependency")
    if first.resource == second.resource and capacities.get(first.resource, 0) < 2:
        reasons.append(f"only one {first.resource}")

    return not reasons, reasons


i0 = IssueInstruction(
    "add x5,x1,x2", frozenset({"x1", "x2"}), frozenset({"x5"}), "ALU"
)
i1 = IssueInstruction(
    "sub x7,x5,x3", frozenset({"x5", "x3"}), frozenset({"x7"}), "ALU"
)

allowed, reasons = can_issue_pair(i0, i1, capacities={"ALU": 2})
assert not allowed and reasons == ["same-group RAW dependency"]
print(reasons)
```

</details>

| Design | Peak issue | Main advantage | Main limitation |
|---|---:|---|---|
| single-cycle scalar | one instruction per long cycle | simple state transition | long clock and no stage overlap |
| five-stage scalar pipeline | one instruction per short cycle | good throughput with understandable control | hazards reduce CPI and bypass paths add timing cost |
| in-order superscalar | multiple instructions in program order | exploits obvious independent neighbors | stalls when the next group contains a dependency |
| out-of-order superscalar | multiple ready instructions chosen dynamically | looks beyond blocked instructions for parallel work | rename, scheduling, recovery, power, and verification complexity |

Pipelining exposes a general performance principle: a fast common path depends on a recovery path that preserves correctness when assumptions fail. Forwarding assumes a value is ready; interlocks handle when it is not. Prediction assumes a PC; flushing repairs a miss. Speculation performs internal work early; ordered retirement prevents that work from violating architectural state.

**Chapter summary.** A five-stage pipeline overlaps IF, ID, EX, MEM, and WB to improve throughput while preserving the sequential ISA contract. Pipeline registers carry data, control, validity, PCs, and exception metadata across clock boundaries. Ideal execution approaches one completion per cycle, but the slowest stage, register overhead, fill time, stalls, and flushes determine actual performance. Structural hazards reflect insufficient hardware capacity. RAW data hazards require forwarding when a value exists and an interlock when it does not; WAR and WAW become important when execution can reorder. Control hazards motivate static and dynamic prediction, while valid bits and flushes suppress wrong-path effects. Precise exception logic completes older work, suppresses the faulting instruction, and removes younger work before trap entry. Superscalar execution extends overlap across multiple instructions per stage and consequently requires more resources, broader dependency checking, and often renaming plus ordered retirement. Chapter 08 will follow the IF and MEM requests into the cache hierarchy, where hit time affects the pipeline clock and misses add variable stall cycles to CPI.